In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 

In [ ]:
def get_dataset_path_list(diretory):
    dataset_path_list = []
    for file in os.listdir(diretory):
        path = os.path.join(diretory, file)
        dataset_path_list.append(path)
    return dataset_path_list

In [ ]:
def decomposicao_svd(df):
    U, S, Vt = np.linalg.svd(df, full_matrices=False)
    return U, S, Vt

def reconstruir_matriz(U, S, Vt, r):
    U_reduced = U[:, :r]
    S_reduced = np.diag(S[:r])
    Vt_reduced = Vt[:r, :]
    matriz_reconstruida = np.dot(U_reduced, np.dot(S_reduced, Vt_reduced))
    return pd.DataFrame(matriz_reconstruida)

def calcular_variabilidade_acumulada(S):
    variabilidade = np.cumsum(S**2) / np.sum(S**2)
    return variabilidade

def grafico_variabilidade(variabilidade):
    plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def svd_suavizacao(df, variabilidade_desejada=0.95):
    U, S, Vt = decomposicao_svd(df)
    variabilidade = calcular_variabilidade_acumulada(S)
    
    # Determina o número de componentes necessários para atingir a variabilidade desejada
    r = np.where(variabilidade >= variabilidade_desejada)[0][0] + 1
    
    # Plota a variabilidade para visualização
    grafico_variabilidade(variabilidade)
    
    # Reconstrói a matriz com r componentes principais
    df_suavizado = reconstruir_matriz(U, S, Vt, r)
    
    return df_suavizado

# Exemplo de uso:
def aplicar_svd_suavizacao(caminho_csv, destino_csv, variabilidade_desejada=0.95):
    # Carrega o arquivo CSV original
    df = pd.read_csv(caminho_csv)
    
    # Preserva a coluna timestamp
    timestamps = df['Timestamp']
    
    # Aplica a suavização com SVD na coluna throughput
    df_suavizado = svd_suavizacao(df[['Throughput']].fillna(0), variabilidade_desejada)
    
    # Reconstrói o DataFrame final com timestamp e throughput suavizado
    df_resultado = pd.DataFrame({
        'Timestamp': timestamps,
        'Throughput': df_suavizado[0]  # Coluna throughput suavizada
    })
    
    # Salva o resultado em um novo arquivo CSV
    df_resultado.to_csv(destino_csv, index=False)
    print(f"Arquivo CSV suavizado gerado em '{destino_csv}'")



In [ ]:
# Caminhos para o CSV de entrada e saída
source_dir = '../datasets/imputed-choosen-best-svd/knn'
dest_dir = '../datasets/smothnization'

for file in os.listdir(source_dir):
    caminho_csv = source_dir + '/' + file
    destino_csv = dest_dir + '/' + file
    aplicar_svd_suavizacao(caminho_csv, destino_csv)